# Class Balancing and Calibration for Toxicity Prediction on New Chemical Scaffolds

**Reproducible Colab notebook — CSC606 Machine Learning individual project**
Muhammad Zohaib Chaudhary

---

## Research question

*Which balancing methods help models find more active compounds on new chemical
scaffolds while keeping their probability estimates reliable?*

Every competing method is given the same screening budget (the top 10% of ranked test
compounds), and the predicted probabilities of each method are examined before and after
sigmoid calibration fitted on a separate calibration partition.

## What this notebook contains

| Section | Contents |
| --- | --- |
| 1 | Title, research question, experimental design, scope caveat |
| 2 | Environment setup (Colab-guarded pip install pinned to `requirements.txt`), then the design grid read from `configs/full.json` |
| 3 | Data provenance: source URL, SHA-256, record counts, cleaning audit |
| 4 | Partition design: realised fractions, positives per partition, scaffold caveat |
| 5 | **Live re-derivation**: the pipeline is actually executed for one partition |
| 6 | Full committed results: seed means and standard deviations, bootstrap deltas |
| 7 | Figures: panels regenerated here plus the committed PNG panels |
| 8 | Similarity analysis over the fixed nearest-neighbour Tanimoto bins |
| 9 | Documented limitations |

## Experimental design

The benchmark is a fixed grid: **three Tox21 assays** (NR-AR, NR-AhR, SR-MMP), **two
classifiers** (regularized logistic regression, random forest), **four balancing choices**
(no balancing, inverse-frequency class weights, random undersampling, SMOTEENN), **two
split kinds** (Bemis-Murcko scaffold hold-out and a stratified random reference split),
and **three fixed seeds** (17, 42, 2026). The exact grid is printed from
`configs/full.json` in section 2.5 rather than restated here from memory.

Each molecule is a 2,048-bit Morgan fingerprint of radius 2 (ECFP4-style). Hyperparameters
are chosen by mean average precision over three cross-validation folds **inside the training
partition only**; resampling and class weights are fitted inside each fold. The calibration
partition and the test partition never inform model or sampler selection.

## Scope caveat

> **An active assay result is an experimental endpoint. It is not a determination that a
> chemical is harmful to people.** The three endpoints here measure androgen-receptor
> activity (NR-AR), aryl-hydrocarbon-receptor activity (NR-AhR), and mitochondrial
> membrane potential (SR-MMP) in laboratory assays. Everything reported below concerns
> model ranking performance, probability calibration, descriptive uncertainty, and the
> limits of the experimental design — nothing more.

## Note on the cited benchmark

The evaluation layout here follows the setup described by the ToxBench study (reference
[4] in the proposal: Kartic, Seo, Yi and Park, 2026). `docs/references.md` records that only
the bibliographic metadata and the publisher abstract of that article were verified, and
that **the proposal's specific assertion about its tree-model class weighting was not
independently verified and is not treated as confirmed anywhere in this notebook.**

## How to run this notebook

* **On Google Colab** — run every cell in order. Section 2 locates the project tree
  (offering a ZIP upload if it is not already present) and then installs the pinned
  dependency versions from the project's own `requirements.txt`. If Colab asks for a
  session restart after the install, restart and run all cells again; the install cell is
  idempotent.
* **Locally** — launch the notebook from inside the project checkout using the project
  virtual environment. The Colab-only cells detect that `google.colab` is absent and skip
  themselves cleanly.

Nothing in this notebook writes to `data/`, `partitions/`, `outputs/full/` or
`outputs/smoke/`. The one cell that executes the pipeline writes to a scratch directory,
`outputs/notebook_demo/`, and asserts that before it starts.

## 2. Environment setup

The next three cells (a) locate the project tree and make it the working directory,
(b) install the pinned dependencies **only when running on Colab**, and (c) import the
project package and report the installed versions against `requirements.txt`.

The Colab-specific work is guarded by a `"google.colab" in sys.modules` check, so running
this notebook locally skips it and keeps using the project virtual environment.

In [ ]:
# --- 2.1 Locate the project tree and make it the working directory --------------------
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = None


def _looks_like_project(path):
    path = Path(path)
    return (path / "toxbench" / "runner.py").is_file() and (path / "configs" / "full.json").is_file()


def activate(path):
    """Make `path` the project root: chdir into it and put it on sys.path."""
    global PROJECT_ROOT
    PROJECT_ROOT = Path(path).resolve()
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    return PROJECT_ROOT


def _descendants(base, depth=2):
    """`base` plus its subdirectories down to `depth` levels, for a bounded search."""
    found = []
    level = [Path(base)]
    for _ in range(depth + 1):
        found.extend(level)
        children = []
        for directory in level:
            try:
                children.extend(sorted(p for p in directory.iterdir()
                                       if p.is_dir() and not p.name.startswith(".")))
            except OSError:
                continue
        level = children
    return found


def find_project_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    if IN_COLAB:
        # A ZIP may unpack with or without a top-level folder, so search a couple of levels.
        for base in (Path("/content"), Path("/content/toxbench_project"),
                     Path("/content/drive/MyDrive")):
            if base.is_dir():
                candidates.extend(_descendants(base, depth=2))
    for candidate in candidates:
        try:
            if _looks_like_project(candidate):
                return candidate
        except OSError:
            continue
    return None


_found = find_project_root()
if _found is not None:
    activate(_found)
    print("Project root :", PROJECT_ROOT)
    print("Working dir  :", Path.cwd())
    print("Running on Google Colab:", IN_COLAB)
elif IN_COLAB:
    print("Project tree not found on this Colab session.")
    print("Run the next cell to upload the project ZIP bundle.")
else:
    raise FileNotFoundError(
        "Could not locate the project root (a directory containing toxbench/runner.py "
        "and configs/full.json) from " + str(Path.cwd()) + " upwards. Start this notebook "
        "from inside the project checkout."
    )

In [ ]:
# --- 2.2 COLAB ONLY: obtain the project tree if it is not already present -------------
if IN_COLAB and PROJECT_ROOT is None:
    from google.colab import files  # noqa: F401  (Colab-only import)

    print("Select the project ZIP bundle to upload (e.g. project606.zip).")
    uploaded = files.upload()
    target = Path("/content/toxbench_project")
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    for name in uploaded:
        shutil.unpack_archive(name, str(target))
    extracted = None
    for candidate in [target, *sorted(p for p in target.rglob("*") if p.is_dir())]:
        if _looks_like_project(candidate):
            extracted = candidate
            break
    if extracted is None:
        raise FileNotFoundError("The uploaded archive does not contain toxbench/runner.py.")
    activate(extracted)
    print("Project root :", PROJECT_ROOT)
elif IN_COLAB:
    print("Project tree already present at", PROJECT_ROOT, "- nothing to upload.")
else:
    print("Not running on Google Colab: skipping the project-upload cell.")

In [ ]:
# --- 2.3 COLAB ONLY: install the pinned dependency versions --------------------------
REQUIREMENTS = PROJECT_ROOT / "requirements.txt"

if IN_COLAB:
    print("Installing pinned dependencies from", REQUIREMENTS)
    print(REQUIREMENTS.read_text().strip())
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS)]
    )
    print("\nInstall finished. If Colab reports that a restart is required, use")
    print("Runtime > Restart session and then run all cells again (this cell is idempotent).")
else:
    print("Not running on Google Colab: skipping the pip install cell.")
    print("Local execution uses the project virtual environment, pinned by")
    print(PROJECT_ROOT / "requirements-lock.txt")

In [ ]:
# --- 2.4 Imports, display helpers, and an installed-versus-pinned version report ------
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd

import toxbench
from toxbench.evaluation import METHODS, CALIBRATIONS, METRIC_NAMES, create_figures, metrics

try:  # IPython is present on Colab and in Jupyter; it may be absent in a bare interpreter.
    from IPython.display import Image as _IPythonImage, display as _ipython_display
    HAS_IPYTHON = True
except ModuleNotFoundError:
    HAS_IPYTHON = False

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 60)


def show(frame, title=None):
    """Render a DataFrame inline on Colab/Jupyter, or as plain text elsewhere."""
    if title:
        print(title)
    data = frame
    if isinstance(data, pd.DataFrame):
        if data.columns.name is not None:
            data = data.rename_axis(columns=None)
        if not isinstance(data.index, pd.MultiIndex) and data.index.name is None:
            data = data.reset_index(drop=True)
    if HAS_IPYTHON:
        _ipython_display(data)
    else:
        print(data.to_string())
    print()


def show_image(path, caption=None):
    """Render a PNG inline on Colab/Jupyter, or report its path elsewhere."""
    path = Path(path)
    if caption:
        print(caption)
    if not path.is_file():
        print("  MISSING:", path)
        return
    if HAS_IPYTHON:
        # The committed panels are 300 dpi, so cap the on-screen width.
        _ipython_display(_IPythonImage(filename=str(path), width=900))
    else:
        print("  [not rendered outside IPython] {} ({:,} bytes)".format(path, path.stat().st_size))


pins = {}
for line in REQUIREMENTS.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        name, _, version = line.partition("==")
        pins[name] = version

rows = []
for name, pinned in pins.items():
    try:
        installed = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        installed = "not installed"
    rows.append({"package": name, "pinned": pinned, "installed": installed,
                 "matches_pin": installed == pinned})
versions = pd.DataFrame(rows)

print("Python  :", sys.version.split()[0])
print("Platform:", sys.platform)
print("toxbench package:", Path(toxbench.__file__).parent)
print()
show(versions, "Installed versions against requirements.txt pins")
print("Packages matching their pin: {} of {}".format(int(versions.matches_pin.sum()), len(versions)))

### 2.5 The experimental design, read from the committed configuration

Rather than restating the design from memory, the cell below prints it from
`configs/full.json`, which is the configuration that produced every committed result in
sections 6 to 8.

In [ ]:
FULL_CONFIG = json.loads((PROJECT_ROOT / "configs" / "full.json").read_text())
SMOKE_CONFIG = json.loads((PROJECT_ROOT / "configs" / "smoke.json").read_text())

ASSAYS = list(FULL_CONFIG["assays"])
SPLIT_KINDS = list(FULL_CONFIG["split_kinds"])
SEEDS = list(FULL_CONFIG["seeds"])
CLASSIFIERS = list(FULL_CONFIG["classifiers"])
BALANCING = list(FULL_CONFIG["balancing"])

design = pd.DataFrame([
    {"design axis": "assays", "count": len(ASSAYS), "values": ", ".join(ASSAYS)},
    {"design axis": "classifiers", "count": len(CLASSIFIERS), "values": ", ".join(CLASSIFIERS)},
    {"design axis": "balancing choices", "count": len(BALANCING), "values": ", ".join(BALANCING)},
    {"design axis": "split kinds", "count": len(SPLIT_KINDS), "values": ", ".join(SPLIT_KINDS)},
    {"design axis": "seeds", "count": len(SEEDS), "values": ", ".join(str(s) for s in SEEDS)},
])
show(design, "Design grid from configs/full.json")

print("Fitted model configurations = {} x {} x {} x {} x {} = {}".format(
    len(ASSAYS), len(CLASSIFIERS), len(BALANCING), len(SPLIT_KINDS), len(SEEDS),
    len(ASSAYS) * len(CLASSIFIERS) * len(BALANCING) * len(SPLIT_KINDS) * len(SEEDS)))
print()
print("Logistic regression C grid :", FULL_CONFIG["logistic_C"])
print("Forest max_depth grid      :", FULL_CONFIG["forest_depth"])
print("Forest min_samples_leaf    :", FULL_CONFIG["forest_min_leaf"])
print("Forest trees               :", FULL_CONFIG["forest_trees"])
print("Cross-validation folds     :", FULL_CONFIG["cv_folds"])
print("Bootstrap replicates       :", FULL_CONFIG["bootstrap_replicates"])
print("Protocol version           :", FULL_CONFIG["protocol_version"])
print()
print("Balancing method order used in every table below:", list(METHODS))
print("Calibration states                              :", list(CALIBRATIONS))
print("Metrics recorded per model                      :", list(METRIC_NAMES))

## 3. Data provenance

The input is the public compressed Tox21 CSV linked by DeepChem's official Tox21 loader.
The download metadata is committed in `data/raw/source.json`; the cell below prints it and
then **recomputes the SHA-256 of the committed archive** and re-reads the archive to count
its records, so the provenance claim is checked rather than quoted.

In [ ]:
source = json.loads((PROJECT_ROOT / "data" / "raw" / "source.json").read_text())
raw_archive = PROJECT_ROOT / "data" / "raw" / "tox21.csv.gz"

print("Source URL          :", source["url"])
print("DeepChem loader     :", source["deepchem_loader_url"])
print("Downloaded (UTC)    :", source["downloaded_at_utc"])
print("Recorded SHA-256    :", source["sha256"])
recomputed = hashlib.sha256(raw_archive.read_bytes()).hexdigest()
print("Recomputed SHA-256  :", recomputed)
print("SHA-256 match       :", recomputed == source["sha256"])
print()
print("Recorded byte count :", "{:,}".format(source["bytes"]))
print("Byte count on disk  :", "{:,}".format(raw_archive.stat().st_size))
print("Recorded records    :", "{:,}".format(source["raw_records"]))

raw_frame = pd.read_csv(raw_archive)
print("Records re-read here:", "{:,}".format(len(raw_frame)))
print("Record count match  :", len(raw_frame) == source["raw_records"])
print()
print("Assay label columns in the file ({}):".format(len(source["assay_label_columns"])))
print(" ", ", ".join(source["assay_label_columns"]))
print("Assays selected for this benchmark:", ", ".join(ASSAYS))

### 3.1 Cleaning and audit counts

`data/audit/assay_counts.csv` is the non-overlapping accounting identity for each assay:
raw observed labels, rows dropped for invalid structures, structures removed for
conflicting duplicate labels, consistent duplicate rows merged, and the resulting clean
structure counts. A missing assay label excludes a record only from that assay; missing
labels are never treated as inactive.

In [ ]:
assay_counts = pd.read_csv(PROJECT_ROOT / "data" / "audit" / "assay_counts.csv")

raw_view = assay_counts[["assay", "raw_records", "raw_observed", "raw_positives",
                         "raw_negatives", "raw_missing"]]
show(raw_view, "Raw label accounting in the downloaded file")

clean_view = assay_counts[["assay", "invalid_structure_observed_rows",
                           "valid_structure_observed_rows", "conflicting_structures",
                           "conflicting_rows", "consistent_duplicate_rows_merged",
                           "clean_structures", "clean_positives", "clean_negatives",
                           "clean_prevalence"]]
show(clean_view, "Cleaning outcome per assay (structure level)")

structure_counts = json.loads(
    (PROJECT_ROOT / "data" / "audit" / "structure_counts.json").read_text())
print("Global structure accounting (data/audit/structure_counts.json):")
for key, value in structure_counts.items():
    print("  {:<40} {}".format(key, value))
print()

verification = json.loads((PROJECT_ROOT / "docs" / "data_verification.json").read_text())
checks = verification["checks"]
print("Independent data audit (docs/data_verification.json):")
print("  partitions checked   :", verification["partitions_checked"])
print("  fingerprints checked :", verification["fingerprints_checked"])
print("  checks passing       : {} of {}".format(sum(bool(v) for v in checks.values()), len(checks)))
failed = [name for name, passed in checks.items() if not passed]
print("  checks failing       :", failed if failed else "none")

## 4. Partition design

Each assay is split independently into approximately 60% training, 20% calibration and
20% test. The **scaffold** design allocates whole non-stereochemical Bemis-Murcko scaffold
groups using only scaffold membership and group size — never activity labels — so no
scaffold appears in more than one partition. The **random** design is a stratified
reference split with the same proportions. All competing methods share identical
partitions.

`partitions/partition_summary.csv` records the realised sizes, fractions, positives and
deviations from the 60/20/20 target for every assay, split kind, seed and partition.

In [ ]:
partition_summary = pd.read_csv(PROJECT_ROOT / "partitions" / "partition_summary.csv")
split_design = json.loads((PROJECT_ROOT / "partitions" / "split_design.json").read_text())

print("Prespecified seeds            :", split_design["seeds"])
print("Target fractions              :", split_design["settings"]["fractions"])
print("Scaffold allocation rule      :", split_design["settings"]["scaffold_assignment"])
print("Cross-validation rule         :", split_design["settings"]["cv"])
print("Infeasible-split policy       :", split_design["settings"]["infeasible"])
print()
print("partition_summary.csv rows    :", len(partition_summary))

PARTITION_ORDER = ["train", "calibration", "test"]
final_parts = partition_summary[partition_summary.partition.isin(PARTITION_ORDER)].copy()
final_parts["partition"] = pd.Categorical(final_parts.partition, PARTITION_ORDER, ordered=True)
final_parts["assay"] = pd.Categorical(final_parts.assay, ASSAYS, ordered=True)

sizes = (final_parts
         .groupby(["assay", "split", "partition"], observed=True)
         .agg(n=("n", "first"),
              realised_fraction=("fraction", "first"),
              target_fraction=("target_fraction", "first"),
              deviation_pp=("deviation_percentage_points", "first"),
              scaffold_groups_min=("scaffolds", "min"),
              scaffold_groups_max=("scaffolds", "max"))
         .reset_index())
show(sizes, "Realised train/calibration/test sizes and fractions (identical across the three seeds)")

In [ ]:
positives = (final_parts
             .pivot_table(index=["assay", "split", "partition"], columns="seed",
                          values="positive", aggfunc="first", observed=True, sort=False)
             .astype(int)
             .reset_index())
positives.columns = [c if isinstance(c, str) else "positives_seed_{}".format(c)
                     for c in positives.columns]
show(positives, "Positive (active) compounds per partition, by seed")

prevalence = (final_parts
              .pivot_table(index=["assay", "split", "partition"], columns="seed",
                           values="prevalence", aggfunc="first", observed=True, sort=False)
              .reset_index())
prevalence.columns = [c if isinstance(c, str) else "prevalence_seed_{}".format(c)
                      for c in prevalence.columns]
show(prevalence, "Realised active fraction per partition, by seed")

exploratory_parts = final_parts[final_parts.exploratory]
print("Final partitions flagged exploratory (fewer than 20 positives):",
      len(exploratory_parts))
if len(exploratory_parts):
    show(exploratory_parts[["assay", "split", "seed", "partition", "n", "positive"]])

### 4.1 The scaffold-split caveat

The scaffold allocator is label-independent and processes groups by decreasing size times
a seeded priority, assigning each group to the partition with the largest remaining
absolute capacity. Two groups are far larger than any other: the single `__EMPTY__` group
that holds every acyclic structure, and the benzene group `c1ccccc1`. Under this fixed
size-based rule, **both land in training for every assay and every seed**, and they also
dominate two of the three training cross-validation folds.

The consequence, computed below rather than asserted, is that a scaffold test partition
describes performance on the remaining held-out scaffold distribution — not on a uniform
random draw of all scaffold groups.

In [ ]:
giant_rows = []
cv_rows = []
for assay in ASSAYS:
    for seed in SEEDS:
        manifest = pd.read_csv(PROJECT_ROOT / "partitions" / assay / "scaffold" / "{}.csv".format(seed),
                               keep_default_na=False)
        grouped = (manifest.groupby("scaffold")
                   .agg(group_size=("structure_id", "size"),
                        partitions=("partition", lambda s: "/".join(sorted(set(s)))))
                   .sort_values("group_size", ascending=False))
        for scaffold, row in grouped.head(2).iterrows():
            giant_rows.append({"assay": assay, "seed": seed,
                               "scaffold": scaffold or "(empty string)",
                               "group_size": int(row.group_size),
                               "partition(s)": row.partitions})
        train_rows = manifest[manifest.partition == "train"]
        for scaffold in ("__EMPTY__", "c1ccccc1"):
            member = train_rows[train_rows.scaffold == scaffold]
            if len(member):
                folds = member.cv_fold.value_counts().sort_index()
                cv_rows.append({"assay": assay, "seed": seed, "scaffold": scaffold,
                                "train_members": len(member),
                                "cv_folds_occupied": ", ".join(
                                    "fold {}: {}".format(int(f), int(c)) for f, c in folds.items())})

giants = pd.DataFrame(giant_rows)
show(giants, "Two largest scaffold groups per assay and seed, and where they are allocated")
print("Giant-group placements landing in training only:",
      int((giants["partition(s)"] == "train").sum()), "of", len(giants))
print()
show(pd.DataFrame(cv_rows), "Cross-validation fold occupancy of the two giant groups inside training")

cv_folds = partition_summary[partition_summary.partition == "cv_validation"]
scaffold_cv = cv_folds[cv_folds.split == "scaffold"]
print("Scaffold CV validation folds:", len(scaffold_cv))
print("Largest absolute deviation from equal thirds: {:.2f} percentage points".format(
    scaffold_cv.deviation_percentage_points.abs().max()))
print("Scaffold CV folds flagged exploratory (fewer than 20 positives):",
      int(scaffold_cv.exploratory.sum()))
show(scaffold_cv[scaffold_cv.exploratory][["assay", "seed", "cv_fold", "n", "positive", "prevalence"]],
     "Scaffold CV validation folds flagged exploratory")

random_cv = cv_folds[cv_folds.split == "random"]
print("Random CV validation folds:", len(random_cv),
      "| largest absolute deviation from equal thirds: {:.3f} percentage points".format(
          random_cv.deviation_percentage_points.abs().max()))

## 5. Live re-derivation of one partition

Sections 6 to 8 read results that were computed once and committed. This section runs the
project's own command line interface **now**, on this machine, so the reader sees real code
produce real numbers.

The demonstration uses `configs/smoke.json`: a single assay (NR-AR), a single split kind
(scaffold), a single seed (17), both classifiers and all four balancing choices — eight
fitted models out of the benchmark's full grid. It is a demonstration slice, not a re-run
of the benchmark.

**Write isolation.** The run is directed at the scratch directory
`outputs/notebook_demo/`. The first cell asserts the resolved output path is not, and is
not inside, `outputs/full/`, `outputs/smoke/`, `data/` or `partitions/`, and then clears
the scratch directory so the reported wall time is a cold run rather than a checkpoint
resume. The `prepare` command is never invoked here, because that is the only command that
writes to `data/` and `partitions/`.

In [ ]:
NOTEBOOK_STARTED = time.time()  # taken here, immediately before this notebook's first write

DEMO_OUTPUT = (PROJECT_ROOT / "outputs" / "notebook_demo").resolve()
PROTECTED = [(PROJECT_ROOT / p).resolve() for p in
             ("outputs/full", "outputs/smoke", "data", "partitions")]

assert DEMO_OUTPUT.name == "notebook_demo", DEMO_OUTPUT
assert PROJECT_ROOT in DEMO_OUTPUT.parents, "demo output must live inside the project"
for protected in PROTECTED:
    assert DEMO_OUTPUT != protected, "demo output must not be a protected directory"
    assert protected not in DEMO_OUTPUT.parents, "demo output must not be inside a protected directory"
    assert DEMO_OUTPUT not in protected.parents, "demo output must not contain a protected directory"

print("Demo output directory :", DEMO_OUTPUT)
print("Protected directories :")
for protected in PROTECTED:
    print("   ", protected)
print()
print("Smoke configuration (configs/smoke.json):")
print(json.dumps(SMOKE_CONFIG, indent=2))
expected_models = (len(SMOKE_CONFIG["assays"]) * len(SMOKE_CONFIG["split_kinds"])
                   * len(SMOKE_CONFIG["seeds"]) * len(SMOKE_CONFIG["classifiers"])
                   * len(SMOKE_CONFIG["balancing"]))
print("\nModels this demonstration will fit:", expected_models)

if DEMO_OUTPUT.exists():
    print("Clearing", DEMO_OUTPUT, "so the timing below is a cold run.")
    shutil.rmtree(DEMO_OUTPUT)

In [ ]:
# --- 5.1 Fit the eight demonstration models through the project CLI -------------------
run_command = [sys.executable, "-m", "toxbench.runner", "run",
               "--root", str(PROJECT_ROOT),
               "--config", str(PROJECT_ROOT / "configs" / "smoke.json"),
               "--output", str(DEMO_OUTPUT)]
print("$", " ".join(run_command))
print()

run_started = time.perf_counter()
run_proc = subprocess.run(run_command, cwd=str(PROJECT_ROOT), text=True, capture_output=True)
run_seconds = time.perf_counter() - run_started

print(run_proc.stdout.strip())
if run_proc.stderr.strip():
    print("--- stderr (last 2000 characters) ---")
    print(run_proc.stderr.strip()[-2000:])
print()
print("Return code              :", run_proc.returncode)
print("Cell wall time (seconds) : {:.1f}".format(run_seconds))
assert run_proc.returncode == 0, "the runner exited non-zero"

# The CLI records per-model failures in failure.json and still exits zero, so check status.
execution_status = json.loads((DEMO_OUTPUT / "execution_status.json").read_text())
print("execution_status.json    :", execution_status)
assert execution_status["failure_files"] == 0, "at least one demonstration model failed"
assert execution_status["completed_models"] == expected_models, "unexpected number of models"
print("\nAll {} demonstration models completed with no failures.".format(
    execution_status["completed_models"]))

In [ ]:
# --- 5.2 Evaluate the demonstration partition through the project CLI ----------------
evaluate_command = [sys.executable, "-m", "toxbench.runner", "evaluate",
                    "--config", str(PROJECT_ROOT / "configs" / "smoke.json"),
                    "--output", str(DEMO_OUTPUT)]
print("$", " ".join(evaluate_command))
print()

evaluate_started = time.perf_counter()
evaluate_proc = subprocess.run(evaluate_command, cwd=str(PROJECT_ROOT), text=True,
                               capture_output=True)
evaluate_seconds = time.perf_counter() - evaluate_started

print(evaluate_proc.stdout.strip())
if evaluate_proc.stderr.strip():
    print("--- stderr (last 2000 characters) ---")
    print(evaluate_proc.stderr.strip()[-2000:])
print()
print("Return code              :", evaluate_proc.returncode)
print("Cell wall time (seconds) : {:.1f}".format(evaluate_seconds))
assert evaluate_proc.returncode == 0, "the evaluator exited non-zero"
print("evaluation_status.json   :",
      json.loads((DEMO_OUTPUT / "evaluation_status.json").read_text()))
print()
print("Total live demonstration wall time: {:.1f} seconds".format(run_seconds + evaluate_seconds))

In [ ]:
# --- 5.3 The numbers this notebook just produced -------------------------------------
demo_metrics = pd.read_csv(DEMO_OUTPUT / "metrics.csv")
demo_view = demo_metrics[demo_metrics.calibration != "constant"].copy()
demo_view["method"] = pd.Categorical(demo_view.method, list(METHODS), ordered=True)
demo_view["calibration"] = pd.Categorical(demo_view.calibration, list(CALIBRATIONS), ordered=True)
demo_view = demo_view.sort_values(["classifier", "method", "calibration"])

columns = ["assay", "split_kind", "seed", "classifier", "method", "calibration",
           "average_precision", "roc_auc", "precision_top10", "recall_top10",
           "brier", "log_loss", "n", "n_positive", "prevalence", "selection_count",
           "exploratory"]
show(demo_view[columns], "Held-out test metrics computed by the cells above (NR-AR, scaffold, seed 17)")

constant_rows = demo_metrics[demo_metrics.calibration == "constant"].drop_duplicates(
    subset=["assay", "split_kind", "seed", "classifier", "calibration"])
show(constant_rows[["classifier", "calibration", "train_prevalence", "average_precision",
                    "roc_auc", "brier", "log_loss", "precision_top10", "recall_top10"]],
     "Reference predictor that always returns the training prevalence")

### 5.4 Does the live slice reproduce the committed benchmark?

`configs/smoke.json` uses the same hyperparameter grids, the same three cross-validation
folds and the same fixed partition as `configs/full.json` for this assay, split kind and
seed; only the bootstrap replicate count differs, and that affects interval width, not the
fitted models. So the eight models just fitted should reproduce the committed predictions
for `NR-AR / scaffold / seed 17`.

The comparison below reports the **maximum absolute difference** in predicted probability
per model, together with whether the two CSV files are byte-identical. Byte identity is
expected on the machine that produced the committed outputs; on different hardware or BLAS
builds small floating-point differences can appear, so this is reported rather than
asserted.

In [ ]:
comparison_rows = []
for classifier in SMOKE_CONFIG["classifiers"]:
    for method in SMOKE_CONFIG["balancing"]:
        relative = Path("runs") / "NR-AR" / "scaffold" / "17" / classifier / method / "predictions.csv"
        live_path = DEMO_OUTPUT / relative
        committed_path = PROJECT_ROOT / "outputs" / "full" / relative
        live = pd.read_csv(live_path, keep_default_na=False)
        committed = pd.read_csv(committed_path, keep_default_na=False)
        same_order = live.structure_id.tolist() == committed.structure_id.tolist()
        same_labels = live.label.tolist() == committed.label.tolist()
        max_difference = max(
            float(np.abs(live[column].to_numpy() - committed[column].to_numpy()).max())
            for column in ("raw", "sigmoid"))
        comparison_rows.append({
            "classifier": classifier, "method": method,
            "test_rows": len(live),
            "same_structure_order": same_order,
            "same_labels": same_labels,
            "max_abs_probability_difference": max_difference,
            "byte_identical_file": (hashlib.sha256(live_path.read_bytes()).hexdigest()
                                    == hashlib.sha256(committed_path.read_bytes()).hexdigest()),
        })

comparison = pd.DataFrame(comparison_rows)
show(comparison, "Live demonstration predictions compared with the committed benchmark predictions")
print("Largest absolute probability difference across all eight models: {:.3e}".format(
    comparison.max_abs_probability_difference.max()))
print("Models whose prediction file is byte-identical to the committed file: {} of {}".format(
    int(comparison.byte_identical_file.sum()), len(comparison)))

In [ ]:
# --- 5.5 A figure produced by the live run -------------------------------------------
demo_panel = DEMO_OUTPUT / "evaluation" / "NR-AR" / "scaffold" / "17" / "forest"
print("Files written by the live evaluation for this group:")
for path in sorted(demo_panel.iterdir()):
    print("  {:<24} {:>10,} bytes".format(path.name, path.stat().st_size))
print()
show_image(demo_panel / "precision_recall.png",
           "Precision-recall panel produced by the live run (NR-AR, scaffold, seed 17, forest)")

## 6. Full benchmark results

Everything in this section is read from the committed outputs of the completed benchmark
under `outputs/full/`. Nothing here is recomputed from the models; the files are the
record of the run.

* `metrics.csv` — one row per fitted model per calibration state.
* `summary.csv` — seed means, standard deviations and counts per
  assay / split kind / classifier / balancing method / calibration state.
* `bootstrap_differences.csv` — paired percentile intervals for each balancing method
  minus the unbalanced baseline, within each individual test partition.

Tables keep the fixed method order `none, weights, undersample, smoteenn` and the fixed
calibration order `raw, sigmoid`. No table is sorted by a metric and no row is highlighted.

In [ ]:
FULL = PROJECT_ROOT / "outputs" / "full"

full_metrics = pd.read_csv(FULL / "metrics.csv")
full_summary = pd.read_csv(FULL / "summary.csv")
bootstrap_differences = pd.read_csv(FULL / "bootstrap_differences.csv")
similarity_metrics = pd.read_csv(FULL / "similarity_metrics.csv")
alignment_audit = pd.read_csv(FULL / "alignment_similarity_audit.csv")

full_execution_status = json.loads((FULL / "execution_status.json").read_text())
full_evaluation_status = json.loads((FULL / "evaluation_status.json").read_text())
full_environment = json.loads((FULL / "environment.json").read_text())

print("outputs/full/metrics.csv                    : {:,} rows".format(len(full_metrics)))
print("outputs/full/summary.csv                    : {:,} rows".format(len(full_summary)))
print("outputs/full/bootstrap_differences.csv      : {:,} rows".format(len(bootstrap_differences)))
print("outputs/full/similarity_metrics.csv         : {:,} rows".format(len(similarity_metrics)))
print("outputs/full/alignment_similarity_audit.csv : {:,} rows".format(len(alignment_audit)))
print()
print("execution_status.json  :", full_execution_status)
print("evaluation_status.json :", full_evaluation_status)
print()
print("Execution environment recorded with the run:")
print("  python   :", full_environment["python"])
print("  platform :", full_environment["platform"])
print("  cpu count:", full_environment["cpu_count"])
for package, version in full_environment["versions"].items():
    print("  {:<17}: {}".format(package, version))
print()
print("Distinct fitted models in metrics.csv :", full_metrics.groupby(
    ["assay", "split_kind", "seed", "classifier", "method"]).ngroups)
print("Calibration states per model          :", sorted(full_metrics.calibration.unique()))
print("Models flagged exploratory (test partition has fewer than 20 positives):",
      int(full_metrics[full_metrics.calibration == "raw"].exploratory.sum()))

### 6.1 Seed means and standard deviations

`summary.csv` aggregates the three seeds for each assay, split kind, classifier, balancing
method and calibration state. Each cell below is `mean ± standard deviation (n = seeds)`.
The standard deviation describes variation across the three fixed seeds; section 9 explains
why three seeds are not independent replication.

Positive direction differs by metric: higher is better for average precision, ROC-AUC and
the precision/recall at the fixed 10% screening budget, while **lower** is better for the
Brier score and log loss.

In [ ]:
HEADLINE_METRICS = ["average_precision", "roc_auc", "precision_top10", "recall_top10",
                    "brier", "log_loss"]


def format_mean_sd(frame, metrics_wanted=HEADLINE_METRICS):
    out = frame[["assay", "split_kind", "classifier", "method", "calibration"]].copy()
    for metric in metrics_wanted:
        mean = frame["{}_mean".format(metric)]
        std = frame["{}_std".format(metric)]
        count = frame["{}_count".format(metric)]
        out[metric] = [
            "undefined" if pd.isna(m) else "{:.4f} +/- {:.4f} (n={})".format(m, 0.0 if pd.isna(s) else s, int(c))
            for m, s, c in zip(mean, std, count)]
    out["prevalence"] = ["{:.4f}".format(v) for v in frame["prevalence_mean"]]
    return out


ordered_summary = full_summary.copy()
ordered_summary["assay"] = pd.Categorical(ordered_summary.assay, ASSAYS, ordered=True)
ordered_summary["split_kind"] = pd.Categorical(ordered_summary.split_kind, SPLIT_KINDS, ordered=True)
ordered_summary["classifier"] = pd.Categorical(ordered_summary.classifier, CLASSIFIERS, ordered=True)
ordered_summary["method"] = pd.Categorical(ordered_summary.method, list(METHODS), ordered=True)
ordered_summary["calibration"] = pd.Categorical(
    ordered_summary.calibration, list(CALIBRATIONS) + ["constant"], ordered=True)
ordered_summary = ordered_summary.sort_values(
    ["assay", "split_kind", "classifier", "method", "calibration"])

model_rows = ordered_summary[ordered_summary.calibration.isin(list(CALIBRATIONS))]
for assay in ASSAYS:
    for split_kind in SPLIT_KINDS:
        block = model_rows[(model_rows.assay == assay) & (model_rows.split_kind == split_kind)]
        show(format_mean_sd(block).drop(columns=["assay", "split_kind"]),
             "{} | {} split | seed mean +/- SD over seeds {}".format(
                 assay, split_kind, ", ".join(str(s) for s in SEEDS)))

In [ ]:
# The constant predictor is the same reference for every balancing method, so it is
# reported once per assay / split kind / classifier rather than repeated four times.
constant_reference = (ordered_summary[ordered_summary.calibration == "constant"]
                      .drop_duplicates(subset=["assay", "split_kind", "classifier"]))
show(format_mean_sd(constant_reference).drop(columns=["method", "calibration"]),
     "Reference predictor that always returns the training prevalence (one row per group)")

### 6.2 What sigmoid calibration did to the ranking

The calibration step fits a sigmoid with a non-negative slope on the separate calibration
partition, so in exact arithmetic it preserves the ordering of compounds. Every fitted
model stores its slope, intercept and a `rank_order_identical` audit flag in
`training.json`; the flag compares the full tie-broken ordering of the test set before and
after calibration, and the cell below reads all of them.

This matters when reading section 6.1: where the ordering is preserved, the ranking metrics
(average precision, ROC-AUC, and precision and recall at the fixed 10% budget) are the same
number before and after calibration. An unchanged ranking metric is **not** a calibration
effect, and the project's documented rule is not to present it as one. The Brier score and
log loss columns are the ones that respond to calibration.

In [ ]:
calibration_rows = []
for path in sorted((FULL / "runs").glob("*/*/*/*/*/training.json")):
    assay, split_kind, seed, classifier, method = path.parts[-6:-1]
    record = json.loads(path.read_text())
    fitted = record["calibration"]
    calibration_rows.append({
        "assay": assay, "split_kind": split_kind, "seed": int(seed),
        "classifier": classifier, "method": method,
        "slope": fitted["slope"], "intercept": fitted["intercept"],
        "calibration_n": fitted["calibration_n"],
        "calibration_positive": fitted["calibration_positive"],
        "constant_mapping": fitted["constant_mapping"],
        "rank_order_identical": record["rank_order_identical"],
    })
calibrations = pd.DataFrame(calibration_rows)

print("Fitted sigmoid calibrators read from outputs/full/runs/.../training.json:", len(calibrations))
print("Smallest fitted slope :", "{:.6f}".format(calibrations.slope.min()))
print("Largest fitted slope  :", "{:.6f}".format(calibrations.slope.max()))
print("Negative slopes (a rank reversal, which would have to be disclosed):",
      int((calibrations.slope < 0).sum()))
print("Degenerate constant mappings:", int(calibrations.constant_mapping.sum()))
print("Models whose full tie-broken ordering of the test set is unchanged by calibration: "
      "{} of {}".format(int(calibrations.rank_order_identical.sum()), len(calibrations)))
print()

exceptions = calibrations[~calibrations.rank_order_identical]
if len(exceptions):
    show(exceptions, "Models where the full tie-broken ordering of the test set differed "
                     "after calibration")

# Check directly in the committed metrics whether any ranking metric actually changed.
keys = ["assay", "split_kind", "seed", "classifier", "method"]
raw_rows = full_metrics[full_metrics.calibration == "raw"].set_index(keys)
sigmoid_rows = full_metrics[full_metrics.calibration == "sigmoid"].set_index(keys)
ranking_metrics = ["average_precision", "roc_auc", "precision_top10", "recall_top10"]
probability_metrics = ["brier", "log_loss"]
comparison_summary = []
for metric in ranking_metrics + probability_metrics:
    difference = (sigmoid_rows[metric] - raw_rows[metric]).abs()
    comparison_summary.append({
        "metric": metric,
        "kind": "ranking" if metric in ranking_metrics else "probability",
        "models_with_identical_value": int((raw_rows[metric] == sigmoid_rows[metric]).sum()),
        "models": len(difference),
        "max_abs_change": "{:.3e}".format(float(difference.max())),
    })
show(pd.DataFrame(comparison_summary),
     "Change from raw to sigmoid predictions, over all committed models")

### 6.3 Balancing minus unbalanced, with paired percentile intervals

`bootstrap_differences.csv` holds, for each individual test partition, the difference
between a balancing method and the unbalanced baseline on exactly the same bootstrap
resamples. Scaffold tests resample whole scaffold groups with replacement; random tests
resample examples. Every comparison within a partition shares identical sampled counts.

These intervals are **within-partition and conditional**: they describe uncertainty of the
already fitted models on the drawn test sample. They do not describe uncertainty from
retraining, they are not independent across the three seeds, and they are not adjusted for
the number of comparisons. Each seed therefore gets its own interval below; interval bounds
are never averaged across seeds.

For the reason given in section 6.2, the `raw` and `sigmoid` rows of a ranking metric carry
the same difference wherever calibration preserved the ordering.

The separate table that follows reports the mean and standard deviation of the *difference*
across the three seeds, which is a different quantity from the intervals.

In [ ]:
bd = bootstrap_differences.copy()
bd["assay"] = pd.Categorical(bd.assay, ASSAYS, ordered=True)
bd["split_kind"] = pd.Categorical(bd.split_kind, SPLIT_KINDS, ordered=True)
bd["classifier"] = pd.Categorical(bd.classifier, CLASSIFIERS, ordered=True)
bd["method"] = pd.Categorical(bd.method, [m for m in METHODS if m != "none"], ordered=True)
bd["calibration"] = pd.Categorical(bd.calibration, list(CALIBRATIONS), ordered=True)
bd["interval"] = ["{:+.4f} [{:+.4f}, {:+.4f}]".format(d, lo, hi)
                  for d, lo, hi in zip(bd.difference, bd.ci_lower, bd.ci_upper)]

print("Bootstrap unit by split kind:")
print(bd.groupby(["split_kind", "bootstrap_unit"], observed=True).size().to_string())
print()
print("Replicates per comparison :", sorted(int(v) for v in bd.n_bootstrap.unique()))
print("Comparisons               : {:,} rows = {} partitions x {} classifiers x {} methods x {} calibrations x {} metrics".format(
    len(bd), len(ASSAYS) * len(SPLIT_KINDS) * len(SEEDS), len(CLASSIFIERS),
    bd.method.nunique(), len(CALIBRATIONS), bd.metric.nunique()))
print("Rows flagged exploratory  :", int(bd.exploratory.sum()))
print()

for metric in HEADLINE_METRICS:
    block = bd[bd.metric == metric].sort_values(
        ["assay", "split_kind", "classifier", "method", "calibration"])
    table = block.pivot(index=["assay", "split_kind", "classifier", "method", "calibration"],
                        columns="seed", values="interval")
    table.columns = ["seed {}".format(c) for c in table.columns]
    show(table.reset_index(),
         "{}: method minus unbalanced, difference with 95% percentile interval, per seed".format(metric))

In [ ]:
seed_spread = (bd.groupby(["assay", "split_kind", "classifier", "method", "calibration", "metric"],
                          observed=True)["difference"]
               .agg(["mean", "std", "count"])
               .reset_index()
               .rename(columns={"mean": "difference_seed_mean",
                                "std": "difference_seed_sd",
                                "count": "seeds"}))
seed_spread = seed_spread[seed_spread.metric.isin(HEADLINE_METRICS)]
seed_spread["metric"] = pd.Categorical(seed_spread.metric, HEADLINE_METRICS, ordered=True)
seed_spread = seed_spread.sort_values(
    ["assay", "split_kind", "classifier", "method", "calibration", "metric"])
seed_spread["difference_seed_mean"] = seed_spread.difference_seed_mean.map("{:+.4f}".format)
seed_spread["difference_seed_sd"] = seed_spread.difference_seed_sd.map("{:.4f}".format)
show(seed_spread, "Mean and standard deviation of the difference across the three seeds "
                  "(separate from, and not a substitute for, the per-seed intervals above)")

undefined = bd[bd.invalid_count > 0]
print("Comparisons with at least one undefined bootstrap replicate:", len(undefined))
if len(undefined):
    show(undefined[["assay", "split_kind", "seed", "classifier", "method", "calibration",
                    "metric", "valid_count", "invalid_count", "n_bootstrap"]],
         "Undefined replicates are excluded per metric and counted here")
print("Complete long-form table: outputs/full/bootstrap_differences.csv ({:,} rows).".format(
    len(bootstrap_differences)))

## 7. Figures

The first cell regenerates a precision-recall panel and a reliability panel **inside this
notebook**, using the project's own `toxbench.evaluation.create_figures` on the committed
`predictions_wide.csv` for one evaluation group. The regenerated files are written to
`outputs/notebook_demo/figures/`, never back into `outputs/full/`.

The cells after that display the committed PNG panel for **every one of the evaluation
groups** — each assay, split kind, seed and classifier. Each precision-recall panel shows
the four balancing methods before and after sigmoid calibration, with the test prevalence
drawn as a dashed reference line. Each reliability panel shows observed active fraction
against mean predicted probability in ten fixed equal-width bins, with the bin counts
plotted underneath on a symmetric-log axis.

In [ ]:
import matplotlib

FIGURE_SOURCE = FULL / "evaluation" / "NR-AR" / "scaffold" / "17" / "forest"
FIGURE_DEST = DEMO_OUTPUT / "figures" / "NR-AR" / "scaffold" / "17" / "forest"
assert DEMO_OUTPUT in FIGURE_DEST.parents, "regenerated figures must stay in the scratch directory"

wide = pd.read_csv(FIGURE_SOURCE / "predictions_wide.csv", keep_default_na=False)
print("Source predictions:", FIGURE_SOURCE / "predictions_wide.csv")
print("Test compounds: {:,} | active: {:,} | prevalence: {:.4f}".format(
    len(wide), int(wide.label.sum()), float(wide.label.mean())))
print("Prediction columns:", [c for c in wide.columns if c.endswith(("_raw", "_sigmoid"))])
print()

previous_backend = matplotlib.get_backend()
create_figures(wide, FIGURE_DEST, "NR-AR | scaffold | seed 17 | forest (regenerated in this notebook)")
try:  # create_figures forces the Agg backend; restore whatever the notebook was using.
    matplotlib.use(previous_backend, force=True)
except Exception as exc:
    print("Could not restore the previous matplotlib backend:", exc)

print("Regenerated into", FIGURE_DEST)
for path in sorted(FIGURE_DEST.iterdir()):
    print("  {:<24} {:>10,} bytes".format(path.name, path.stat().st_size))
print()
show_image(FIGURE_DEST / "precision_recall.png", "Precision-recall panel regenerated here")
show_image(FIGURE_DEST / "reliability.png", "Reliability panel regenerated here")

In [ ]:
# Cross-check the regenerated reliability table against the committed one.
regenerated_bins = pd.read_csv(FIGURE_DEST / "reliability.csv")
committed_bins = pd.read_csv(FIGURE_SOURCE / "reliability.csv")
key = ["method", "calibration", "bin"]
merged = regenerated_bins.merge(committed_bins, on=key, suffixes=("_new", "_committed"))
print("Reliability bin rows compared:", len(merged))
for column in ("count", "positives"):
    print("  identical {:<10}: {}".format(
        column, bool((merged[column + "_new"] == merged[column + "_committed"]).all())))
for column in ("mean_prediction", "observed_fraction"):
    difference = (merged[column + "_new"] - merged[column + "_committed"]).abs()
    print("  max abs difference in {:<18}: {:.3e}".format(column, float(difference.max(skipna=True))))
print()

empty_bins = committed_bins[committed_bins["count"] == 0]
print("Committed reliability bins for this group that contain no test compounds:", len(empty_bins))
print("Committed reliability bins flagged exploratory (fewer than 20 positives): {} of {}".format(
    int(committed_bins.exploratory.sum()), len(committed_bins)))
show(committed_bins[committed_bins.method == "none"][
    ["method", "calibration", "bin", "lower", "upper", "count", "positives",
     "mean_prediction", "observed_fraction", "exploratory"]],
     "Committed reliability bins for the unbalanced baseline of this group")

In [ ]:
# --- Every committed precision-recall panel: 3 assays x 2 splits x 3 seeds x 2 classifiers
EVALUATION_GROUPS = [(assay, split_kind, seed, classifier)
                     for assay in ASSAYS
                     for split_kind in SPLIT_KINDS
                     for seed in SEEDS
                     for classifier in CLASSIFIERS]
print("Committed evaluation groups to display:", len(EVALUATION_GROUPS))
print()
for assay, split_kind, seed, classifier in EVALUATION_GROUPS:
    panel = FULL / "evaluation" / assay / split_kind / str(seed) / classifier
    show_image(panel / "precision_recall.png",
               "Committed precision-recall panel: {} | {} split | seed {} | {}".format(
                   assay, split_kind, seed, classifier))

In [ ]:
# --- Every committed reliability panel ------------------------------------------------
for assay, split_kind, seed, classifier in EVALUATION_GROUPS:
    panel = FULL / "evaluation" / assay / split_kind / str(seed) / classifier
    show_image(panel / "reliability.png",
               "Committed reliability panel: {} | {} split | seed {} | {}".format(
                   assay, split_kind, seed, classifier))

In [ ]:
# --- Inventory of every committed figure file -----------------------------------------
inventory = []
for group in sorted((FULL / "evaluation").glob("*/*/*/*")):
    if not group.is_dir():
        continue
    assay, split_kind, seed, classifier = group.parts[-4:]
    row = {"assay": assay, "split_kind": split_kind, "seed": int(seed), "classifier": classifier}
    for name in ("precision_recall.png", "precision_recall.pdf",
                 "reliability.png", "reliability.pdf"):
        row[name] = (group / name).is_file()
    row["displayed_above"] = (assay, split_kind, int(seed), classifier) in EVALUATION_GROUPS
    inventory.append(row)

inventory = pd.DataFrame(inventory)
inventory["assay"] = pd.Categorical(inventory.assay, ASSAYS, ordered=True)
inventory["split_kind"] = pd.Categorical(inventory.split_kind, SPLIT_KINDS, ordered=True)
inventory = inventory.sort_values(["assay", "split_kind", "seed", "classifier"])
show(inventory, "Every committed evaluation group and its figure files")
print("Evaluation groups       :", len(inventory))
print("Figure files committed  :", int(inventory[["precision_recall.png", "precision_recall.pdf",
                                                  "reliability.png", "reliability.pdf"]].sum().sum()))
print("Groups displayed inline :", int(inventory.displayed_above.sum()))
print("The PDF version of every panel is on disk beside the PNG, under")
print("outputs/full/evaluation/<assay>/<split kind>/<seed>/<classifier>/.")

## 8. Nearest-neighbour similarity analysis

Holding scaffolds out does not guarantee that test fingerprints are dissimilar to the
training set. For every test compound, the pipeline records the maximum Tanimoto similarity
of its Morgan fingerprint to any training compound, and then recomputes all metrics inside
four similarity bins: [0, 0.4), [0.4, 0.6), [0.6, 0.8) and [0.8, 1.0]. The edges are
hard-coded constants in `toxbench/postprocess.py`, not parameters; `docs/references.md`
records the rule that similarity thresholds are fixed before outcomes are viewed.

Bins behave differently from the full test set: a bin containing **no active compounds
yields an undefined average precision and ROC-AUC by design** — the pipeline records them
as missing rather than substituting zero — and any subset with fewer than 20 actives is
flagged exploratory.

In [ ]:
print("Fixed similarity bins present in similarity_metrics.csv:")
bin_edges = (similarity_metrics[["similarity_low", "similarity_high"]]
             .drop_duplicates().sort_values("similarity_low"))
for low, high in bin_edges.itertuples(index=False):
    print("  [{:.1f}, {:.1f}{}".format(low, high, "]" if high >= 1.0 else ")"))
print()

similarity_metrics["bin"] = [
    "[{:.1f}, {:.1f}{}".format(low, high, "]" if high >= 1.0 else ")")
    for low, high in zip(similarity_metrics.similarity_low, similarity_metrics.similarity_high)]
BIN_ORDER = list(bin_edges.assign(label=[
    "[{:.1f}, {:.1f}{}".format(low, high, "]" if high >= 1.0 else ")")
    for low, high in zip(bin_edges.similarity_low, bin_edges.similarity_high)]).label)

show(alignment_audit.assign(
        assay=pd.Categorical(alignment_audit.assay, ASSAYS, ordered=True),
        split_kind=pd.Categorical(alignment_audit.split_kind, SPLIT_KINDS, ordered=True))
     .sort_values(["assay", "split_kind", "seed", "classifier"]),
     "Per-group test size, actives, and nearest-training-compound similarity "
     "(outputs/full/alignment_similarity_audit.csv)")

In [ ]:
# Bin occupancy is a property of the test partition, identical for every balancing method,
# so it is read once from the unbalanced raw rows.
occupancy = similarity_metrics[(similarity_metrics.method == "none")
                               & (similarity_metrics.calibration == "raw")].copy()
occupancy["assay"] = pd.Categorical(occupancy.assay, ASSAYS, ordered=True)
occupancy["split_kind"] = pd.Categorical(occupancy.split_kind, SPLIT_KINDS, ordered=True)
occupancy["bin"] = pd.Categorical(occupancy["bin"], BIN_ORDER, ordered=True)
occupancy = occupancy[occupancy.classifier == CLASSIFIERS[0]]

counts_table = occupancy.pivot(index=["assay", "split_kind", "seed"], columns="bin", values="n")
positives_table = occupancy.pivot(index=["assay", "split_kind", "seed"], columns="bin",
                                  values="n_positive")
show(counts_table.reset_index(), "Test compounds per fixed similarity bin")
show(positives_table.reset_index(), "Active compounds per fixed similarity bin")

print("Similarity-bin result rows                    : {:,}".format(len(similarity_metrics)))
print("Rows whose bin contains no actives            : {:,}".format(
    int((similarity_metrics.n_positive == 0).sum())))
print("Rows with undefined average precision         : {:,}".format(
    int(similarity_metrics.average_precision.isna().sum())))
print("Rows with undefined ROC-AUC                   : {:,}".format(
    int(similarity_metrics.roc_auc.isna().sum())))
print("Undefined rows that are exactly the zero-active rows:",
      bool((similarity_metrics.average_precision.isna()
            == (similarity_metrics.n_positive == 0)).all()))
print("Rows flagged exploratory (fewer than 20 actives): {:,} of {:,}".format(
    int(similarity_metrics.exploratory.sum()), len(similarity_metrics)))

In [ ]:
sim = similarity_metrics.copy()
sim["assay"] = pd.Categorical(sim.assay, ASSAYS, ordered=True)
sim["split_kind"] = pd.Categorical(sim.split_kind, SPLIT_KINDS, ordered=True)
sim["classifier"] = pd.Categorical(sim.classifier, CLASSIFIERS, ordered=True)
sim["method"] = pd.Categorical(sim.method, list(METHODS), ordered=True)
sim["calibration"] = pd.Categorical(sim.calibration, list(CALIBRATIONS), ordered=True)
sim["bin"] = pd.Categorical(sim["bin"], BIN_ORDER, ordered=True)

for metric in ("average_precision", "recall_top10"):
    aggregated = (sim.groupby(["assay", "split_kind", "classifier", "method", "calibration", "bin"],
                              observed=True)
                  .agg(seed_mean=(metric, "mean"),
                       seeds_defined=(metric, "count"),
                       actives_min=("n_positive", "min"),
                       actives_max=("n_positive", "max"))
                  .reset_index())
    aggregated["value"] = [
        "undefined" if defined == 0 else "{:.4f} (seeds={}, actives {}-{})".format(mean, defined, lo, hi)
        for mean, defined, lo, hi in zip(aggregated.seed_mean, aggregated.seeds_defined,
                                         aggregated.actives_min, aggregated.actives_max)]
    table = aggregated.pivot(index=["assay", "split_kind", "classifier", "method", "calibration"],
                             columns="bin", values="value")
    show(table.reset_index(),
         "{} by fixed similarity bin, averaged over the seeds where it is defined".format(metric))

print("Complete per-seed table: outputs/full/similarity_metrics.csv ({:,} rows).".format(
    len(similarity_metrics)))

## 9. Limitations

These are the limitations documented for this benchmark in `docs/data_methods.md`,
`docs/references.md` and `docs/implementation_plan.md`. The cell after this list prints the
counts that each item refers to, straight from the committed files.

**Design and replication**

1. **Three seeds are not independent replication.** The seeds reshuffle the same cleaned
   compounds, so the resulting test partitions overlap in chemistry and are not independent
   datasets. Seed standard deviations describe split and fit variation within one dataset;
   they do not support population-level inference, and they say nothing about generalisation
   to other assays.
2. **The scaffold allocation is fixed and size-driven.** The giant `__EMPTY__` and benzene
   groups land in training for every assay and seed (section 4.1), so a scaffold test
   partition describes performance on the remaining held-out scaffold distribution, not a
   uniform random draw of scaffold groups. The same giant groups dominate two of the three
   training cross-validation folds. `docs/data_methods.md` records this as a material
   limitation for hyperparameter stability and across-seed interpretation. No group was
   split or moved to improve these figures.
3. **Some scaffold cross-validation validation folds carry very few actives**, for every
   seed, and are flagged exploratory. Scaffold cross-validation fold fractions deviate from
   equal thirds by several percentage points, while the final 60/20/20 partitions are
   within a small fraction of a percentage point of target. The cell below prints both
   figures from the committed partition summary.

**Uncertainty**

4. **The bootstrap intervals are descriptive and conditional.** They condition on the
   already fitted models and on the current test partition. They do not quantify
   uncertainty from retraining, from alternative partitions or from the choice of assay.
5. **The intervals are not multiplicity-adjusted.** The benchmark makes a large number of
   paired comparisons across assays, split kinds, classifiers, methods, calibration states
   and metrics. An interval that excludes zero here is a descriptive observation about one
   partition, not a multiplicity-controlled significance claim.
6. **Intervals are per partition and must not be pooled.** Section 6.2 reports each seed's
   interval separately; averaging interval bounds across seeds would not be a valid interval.
7. **Undefined metric values are recorded, not imputed.** Average precision and ROC-AUC are
   undefined for a resample or a subset with a single class, and the pipeline reports valid
   and invalid replicate counts rather than substituting zero.

**Small positive counts**

8. **Any subset with fewer than 20 actives is exploratory**, and the committed files carry an
   explicit `exploratory` flag for exactly that reason. This applies to several similarity
   bins, to reliability bins, and to some cross-validation folds.

**Data and measurement**

9. **Assay labels are noisy, and the three endpoints are experimental.** An active result in
   NR-AR, NR-AhR or SR-MMP is a laboratory measurement, **not a determination that a chemical
   is harmful to people**.
10. **Fingerprint similarity is a limited measure of chemical relatedness.** Morgan
    fingerprints ignore stereochemistry even though structure identity here preserves it,
    tautomers are not canonicalised so residual near-duplicates can remain, and scaffold
    separation does not imply that all test fingerprints have low similarity to training
    examples. Section 8 reports the similarity distributions and the per-bin results for
    that reason.
11. **Synthetic SMOTEENN rows are numerical training vectors, not new molecules**, and a
    1:1 SMOTE target does not guarantee 1:1 counts after the nearest-neighbour cleaning step.

**Scope of the literature claims**

12. **Reference [4] is not treated as confirmed.** Only the bibliographic metadata and the
    publisher abstract of the ToxBench article were verified; `docs/references.md` records
    that the proposal's specific assertion about its tree-model class weighting could not be
    verified from the accessible article text, and it is not asserted here.
13. **The cited balancing studies are motivation, not baselines.** Their curation,
    descriptors, tuning and evaluation differ from this benchmark, so their numbers are not
    directly comparable to these.

**Execution**

14. **Local CPU execution replaced the proposed Colab CPU execution** for the full
    benchmark; this notebook supports Colab reproduction of the committed results and of a
    live slice. The initially installed RDKit 2026.09.2 failed basic Murcko-scaffold
    extraction on the execution host, so RDKit was pinned to 2025.09.3 before any analysis.

**This notebook does not interpret the results.** It describes what each cell computes and
displays. Which balancing choice to prefer, and under what conditions, is a question for the
project report, which reads these same committed tables.

In [ ]:
# --- Counts behind the limitation list, read from the committed files -----------------
print("Seeds used                                   :", SEEDS)
print("Fitted models                                :", full_execution_status["completed_models"])
print("Model failures                               :", full_execution_status["failure_files"])
print("Evaluation groups                            :", full_evaluation_status["partition_classifier_groups"])
print("Bootstrap replicates per comparison          :", full_evaluation_status["bootstrap_replicates"])
print()
print("Giant scaffold groups allocated to training  : {} of {} placements".format(
    int((giants["partition(s)"] == "train").sum()), len(giants)))
print("Scaffold CV validation folds flagged exploratory:", int(scaffold_cv.exploratory.sum()))
print("Smallest positive count in a scaffold CV validation fold:", int(scaffold_cv.positive.min()))
print("Largest scaffold CV deviation from equal thirds: {:.2f} percentage points".format(
    scaffold_cv.deviation_percentage_points.abs().max()))
print("Largest final-partition deviation from 60/20/20: {:.3f} percentage points".format(
    final_parts.deviation_percentage_points.abs().max()))
print()
print("Paired comparisons in bootstrap_differences.csv: {:,}".format(len(bootstrap_differences)))
print("  of which flagged exploratory                 : {:,}".format(int(bootstrap_differences.exploratory.sum())))
print("  with at least one undefined replicate        : {:,}".format(
    int((bootstrap_differences.invalid_count > 0).sum())))
print()
print("Similarity-bin rows                          : {:,}".format(len(similarity_metrics)))
print("  flagged exploratory                        : {:,}".format(int(similarity_metrics.exploratory.sum())))
print("  with no actives, so undefined AP and AUC   : {:,}".format(
    int((similarity_metrics.n_positive == 0).sum())))
print()
print("Test partitions with fewer than 20 actives   :", int(
    full_metrics[full_metrics.calibration == "raw"]
    .drop_duplicates(subset=["assay", "split_kind", "seed"]).exploratory.sum()))
print()
print("Reminder: assay activity is an experimental endpoint, not a determination of human toxicity.")

---

### Files this notebook read and wrote

Read only, from the committed project tree:
`data/raw/source.json`, `data/raw/tox21.csv.gz`, `data/audit/assay_counts.csv`,
`data/audit/structure_counts.json`, `docs/data_verification.json`,
`partitions/partition_summary.csv`, `partitions/split_design.json`,
`partitions/<assay>/scaffold/<seed>.csv`, `configs/full.json`, `configs/smoke.json`,
`requirements.txt`, and everything under `outputs/full/`.

Written, all inside the scratch directory `outputs/notebook_demo/`: the eight
demonstration models and their evaluation in section 5, and the regenerated figure panel in
section 7. The final cell confirms that nothing outside that scratch directory was created
by this notebook.

In [ ]:
print("Scratch directory used by this notebook:", DEMO_OUTPUT)
written = sorted(p for p in DEMO_OUTPUT.rglob("*") if p.is_file())
print("Files written there: {:,}".format(len(written)))
print("Top-level entries:")
for path in sorted(DEMO_OUTPUT.iterdir()):
    print("  ", path.name + ("/" if path.is_dir() else ""))
print()

# Nothing above should have modified a file in a protected directory. Compare every file's
# modification time with the moment this notebook started, in section 2.1.
print("Files modified since this notebook started, per protected directory:")
untouched = True
for protected in PROTECTED:
    recent = [p for p in protected.rglob("*")
              if p.is_file() and p.stat().st_mtime > NOTEBOOK_STARTED]
    untouched = untouched and not recent
    print("  {:<58} {:>5} modified   (files scanned: {:,})".format(
        str(protected.relative_to(PROJECT_ROOT)), len(recent),
        sum(1 for p in protected.rglob("*") if p.is_file())))
    for path in recent[:10]:
        print("      ", path)
print()
print("All protected directories untouched by this notebook:", untouched)
print("Notebook started at (epoch seconds): {:.3f}".format(NOTEBOOK_STARTED))